# Pothole Detection — YOLO Training di Google Colab T4

Target: deteksi jalan berlubang dari video (MP4/YouTube) + siap integrasi web app.

**Kenapa setting ini pas buat T4 gratis:** model kecil (YOLOv8n), imgsz 640, batch 16, epochs 50-100, AMP on, cache off.
T4 = 16GB VRAM, limit waktu ~ cocok untuk fine-tuning, BUKAN training dari nol.

Alur: cek GPU → install → siapkan dataset → train → validasi → test di video kamu → simpan best.pt

In [ ]:
# 0. Cek GPU (pastikan Colab -> Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# 1. Install (Colab sudah ada torch + cv2, tinggal ultralytics)
%pip install -q ultralytics roboflow
import cv2, sys
print('cv2', cv2.__version__)
from ultralytics import YOLO
print('ultralytics ok')

In [ ]:
# 2. Setup folder + (opsional) mount Drive buat simpan best.pt
from google.colab import drive
drive.mount('/content/drive')

import pathlib
ROOT = pathlib.Path('/content/pothole')
(ROOT / 'datasets').mkdir(parents=True, exist_ok=True)
(ROOT / 'runs').mkdir(parents=True, exist_ok=True)
print(ROOT)

## 3. Siapkan dataset (PILIH SALAH SATU)

**Opsi A (disarankan buat T4 + project kuliah): dataset publik pothole.**
Contoh: RDD2022 / Roboflow pothole. Sudah ada anotasi YOLO.

**Opsi B: dataset custom dari 2 video kamu.** Video mentah TANPA label tidak bisa langsung buat training. Harus: extract frame → label manual (Roboflow/LabelImg) → zip → upload.

Jalankan SATU opsi saja.

In [ ]:
# 3A. Opsi A — dataset publik single-class (disarankan, ringan di T4)
# Rekomendasi: varadpisale/pothole-detection-dataset-annotated-yolov8 (Kaggle)
# 8236 train / 1327 valid / 559 test, 1 kelas: pothole. Format YOLOv8.
# Link: https://www.kaggle.com/datasets/varadpisale/pothole-detection-dataset-annotated-yolov8
# Cara: download zip dari Kaggle -> upload ke /content/pothole/datasets/ -> unzip.
# Alternatif multi-negara: RDD2022 https://www.kaggle.com/datasets/aliabdelmenam/rdd-2022 (11GB, 5 kelas)
# atau Roboflow: https://universe.roboflow.com/yolov8-study/pothole-detection-rdd2022

# --- TEMPLATE Roboflow (isi API KEY kamu) ---
# from roboflow import Roboflow
# rf = Roboflow(api_key="ISI_API_KEY")
# project = rf.workspace("WS").project("pothole")
# ds = project.version(1).download("yolov8", location=str(ROOT / 'datasets' / 'pothole_rf'))

# --- Unzip manual kalau upload zip via Files panel: ---
# !unzip -q /content/pothole-detection-dataset-annotated-yolov8.zip -d /content/pothole/datasets/pothole

# --- Cek struktur yang diharapkan: ---
# datasets/pothole/
#   train/images/*.jpg  train/labels/*.txt
#   valid/images/*.jpg  valid/labels/*.txt
#   data.yaml
!ls -R /content/pothole/datasets | head -n 100

In [ ]:
# 3B. Opsi B — extract frame dari video kamu buat dilabeli manual
# Upload dulu 2 mp4 kamu ke /content/pothole/videos/ (via Files panel Colab), lalu jalankan ini.
import cv2, pathlib
VID_DIR = pathlib.Path('/content/pothole/videos')
OUT_DIR = pathlib.Path('/content/pothole/datasets/custom/images_raw')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def extract_frames(video_path, out_dir, every_n=10, max_frames=300):
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    i, saved = 0, 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % every_n == 0:
            cv2.imwrite(str(out_dir / f"{video_path.stem}_{saved:04d}.jpg"), frame)
            saved += 1
            if saved >= max_frames:
                break
        i += 1
    cap.release()
    print(video_path.name, f"fps={fps:.1f}", f"saved={saved}")

if VID_DIR.exists():
    for v in VID_DIR.glob('*.mp4'):
        extract_frames(v, OUT_DIR, every_n=15, max_frames=300)
else:
    print('Upload dulu mp4 ke', VID_DIR)

In [ ]:
# 4. data.yaml — sesuaikan path setelah dataset siap
import pathlib, textwrap
ROOT = pathlib.Path('/content/pothole')
# GANTI path ini sesuai datasetmu (contoh Opsi A):
DATA_YAML = ROOT / 'datasets' / 'pothole_rf' / 'data.yaml'
print(DATA_YAML, DATA_YAML.exists())
if DATA_YAML.exists():
    print(DATA_YAML.read_text())
else:
    print(textwrap.dedent('''\
    Contoh data.yaml minimal (1 kelas):
    path: /content/pothole/datasets/pothole
    train: train/images
    val: valid/images
    names:
      0: pothole
    '''))

In [ ]:
# 5. TRAIN — setting hemat T4
# yolov8n.pt = paling ringan, cocok T4 gratis. Naik ke yolov8s.pt kalau mAP kurang dan VRAM masih sisa.
# epochs 50 dulu (1-2 jam). Lanjut 100 kalau belum overfit.
from ultralytics import YOLO

DATA = '/content/pothole/datasets/pothole_rf/data.yaml'  # <-- GANTI sesuai punyamu
model = YOLO('yolov8n.pt')

model.train(
    data=DATA,
    epochs=50,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,
    amp=True,
    cache=False,
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    project='/content/pothole/runs',
    name='yolov8n_pothole_T4',
    exist_ok=True,
    plots=True,
)

In [ ]:
# 6. Validasi — lihat mAP50, mAP50-95, precision, recall
from ultralytics import YOLO
BEST = '/content/pothole/runs/yolov8n_pothole_T4/weights/best.pt'
m = YOLO(BEST)
print(m.val(data=DATA, imgsz=640, device=0))

In [ ]:
# 7. Test di video kamu (upload mp4 ke /content/pothole/videos/test.mp4)
from ultralytics import YOLO
BEST = '/content/pothole/runs/yolov8n_pothole_T4/weights/best.pt'
m = YOLO(BEST)
# image / video / youtube URL juga bisa
m.predict(
    source='/content/pothole/videos/test.mp4',  # ganti ke videomu
    imgsz=640, conf=0.35, iou=0.5, device=0,
    save=True, save_txt=True, project='/content/pothole/runs', name='pred_test', exist_ok=True
)
print('selesai, cek folder runs/pred_test')

In [ ]:
# 8. Simpan best.pt ke Drive + download lokal
import shutil, pathlib
BEST = pathlib.Path('/content/pothole/runs/yolov8n_pothole_T4/weights/best.pt')
DST = pathlib.Path('/content/drive/MyDrive/pothole_best.pt')
if BEST.exists():
    shutil.copy(BEST, DST)
    print(f"tersimpan: {DST} ({DST.stat().st_size/1e6:.1f} MB)")
else:
    print('best.pt belum ada, cek training')
from google.colab import files
# files.download(str(BEST))  # uncomment buat download langsung

## Catatan T4 (penting)

- Kalau `CUDA out of memory`: turunkan `batch` ke 8, `imgsz` ke 512.
- Kalau Colab disconnect: centang `Edit > Notebook settings > GPU`, simpan checkpoint tiap epoch (default sudah), resume pakai `model = YOLO('runs/.../weights/last.pt')` lalu `model.train(resume=True)`.
- Video YouTube umumnya **tanpa GPS** → deteksi tetap jalan, tapi peta butuh koordinat manual/eksternal. Jangan klaim bbox = GPS.
- Target realistis awal: mAP50 0.5-0.7 di dataset publik, lalu uji di video kamu.